# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

We will explore its structure, access records by their `@id`s, and perform data processing and visualization.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via the croissant schema URL
dataset = mlc.Dataset(croissant_url)

# Display dataset-level metadata (name and description)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and their corresponding fields and `@id` identifiers.

In [ ]:
# List available record sets and their fields by @id
print("Available record sets (@id, name, description):\n")
record_sets = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '-')}")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    # List fields for each record_set
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - @id: {fld.id}\n      Name: {getattr(fld, 'name', '-')} | Data Type: {getattr(fld, 'data_type', '-')} | Description: {getattr(fld, 'description', '-')} ")
    else:
        print("  No fields defined.")
    record_sets.append(rs.id)
    print("")

# Preview the first record for each record set
for rs_id in record_sets:
    print(f"\nSample record for RecordSet @id: {rs_id}")
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        break

## 3. Data Extraction

Load full records from record sets into pandas DataFrames for further analysis. Only public and tabular record sets are extracted here.

In [ ]:
# Create a dictionary of DataFrames for each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '{rs_id}' with columns: {df.columns.tolist()}")

# Show info for the main tabular record set (usually the largest table)
main_recordset_id = None
max_rows = 0
for rs_id, df in dataframes.items():
    if df.shape[0] > max_rows:
        max_rows = df.shape[0]
        main_recordset_id = rs_id

if main_recordset_id is not None:
    print(f"\nMain record set selected: {main_recordset_id}")
    print(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)

Filter and transform data. We will select a numeric field (e.g., patient age or diagnosis interval), remove outliers, normalize the field, and group data by a categorical field (e.g., sex or MSI status).

In [ ]:
# Show columns for main record set
print(f"Columns in main record set ({main_recordset_id}):")
print(dataframes[main_recordset_id].columns.tolist())

# Based on actual column names, select fields by their @id
# (Replace the below '@id's and column keys with those found in your dataset overview)
# Example for this dataset (adjust as needed):
numeric_field_id = None
group_field_id = None
# Attempt to detect a likely numeric field
for col in dataframes[main_recordset_id].columns:
    if 'age' in col.lower():
        numeric_field_id = col
    elif ('interval' in col.lower() or 'years' in col.lower()) and numeric_field_id is None:
        numeric_field_id = col
    if any(x in col.lower() for x in ['sex','msi','status','gender']) and group_field_id is None:
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = dataframes[main_recordset_id].columns[0]  # fallback

if group_field_id is None:
    group_field_id = dataframes[main_recordset_id].columns[-1]  # fallback

print(f"\nUsing numeric field (@id): {numeric_field_id}")
print(f"Using group field (@id): {group_field_id}")

# Clean: convert numeric field to float, drop rows if not convertable
df = dataframes[main_recordset_id]
df = df.copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
df = df.dropna(subset=[numeric_field_id])

# Remove outliers (simple: keep between 1st and 99th percentile)
minv, maxv = df[numeric_field_id].quantile([0.01, 0.99])
filtered_df = df[(df[numeric_field_id] >= minv) & (df[numeric_field_id] <= maxv)]

print(f"Filtered records for {numeric_field_id} between {minv:.1f} and {maxv:.1f}: {len(filtered_df)} rows")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group field and calculate mean of the numeric field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=10)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group field (if categorical with few unique values)
if filtered_df[group_field_id].nunique() <= 10:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset using the Croissant schema and the `mlcroissant` library. We reviewed both record set and field structure using their `@id`s, extracted records into DataFrames, performed basic filtering, normalization, and grouping, and visualized key features. 

For in-depth analysis, consult the Croissant metadata for additional field details, or extend your exploration with more sophisticated visualizations and modeling.